In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import datetime
import requests
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup, Comment
from urllib.parse import urljoin

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MV MMA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running MV MMA Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

processdate = now.strftime('%Y-%m-%d')

# Lists from Jira DECD-6136 (ticket skips ListNr 2).
# mma.gov.mv is an AngularJS SPA: the #/ routes load static partial HTML files that are
# directly fetchable - no browser needed. The brokers list (4) is a date-stamped PDF whose
# current link is resolved from the insurance-sector partial each run.
BASE = 'https://www.mma.gov.mv/'
regdict = {
    1: {"ListName": "Register of Banks",
        "URL": "http://www.mma.gov.mv/#/financialstability/bankingsector/registerofbanks",
        "Partial": "partials/financialStability/registerofbanks.html"},
    3: {"ListName": "Register of Insurance Providers",
        "URL": "http://www.mma.gov.mv/#/financialstability/insurancesector/insuranceproviders",
        "Partial": "partials/financialStability/insuranceProvidors.html"},
    4: {"ListName": "Register of Insurance Brokers",
        "URL": "https://www.mma.gov.mv/files/financialstability/insurance-brokers-03052026.pdf",
        "Partial": "partials/financialStability/insurancesector.html"},
    5: {"ListName": "Register of Licensed Payment Service Providers",
        "URL": "https://www.mma.gov.mv/#/bankingandpayments/registerofserviceproviders",
        "Partial": "partials/paymentsinfrastructure/registerofserviceproviders.html"},
}

ListLabeldict = {1: 1, 3: 2, 4: 2, 5: 4}

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def get_soup(path):
    r = requests.get(urljoin(BASE, path), headers=HEADERS, timeout=60, verify=False)
    r.raise_for_status()
    r.encoding = 'utf-8'   # server omits charset; requests would fall back to latin-1 and mangle 'Malé'
    soup = BeautifulSoup(r.text, 'html.parser')
    for com in soup.find_all(string=lambda t: isinstance(t, Comment)):
        com.extract()   # partials keep former entities (e.g. HSBC) inside HTML comments
    return soup

def clean_lines(el):
    return [re.sub(r'\s+', ' ', l).strip() for l in el.get_text('\n').split('\n') if l.strip()]

def add_row(rowdata):
    for key in sqldict:
        sqldict[key].append(rowdata.get(key, ''))

def common_fields(listnr):
    return {'ListLabel': ListLabeldict[listnr],
            'RegCtry': 'MV',
            'RegCode': 'MMA',
            'ListCode': str(listnr),
            'ListName': regdict[listnr]['ListName'],
            'ListLanguage': 'EN',
            'ListProcessDate': processdate,
            'RegulationType': 'Regulated',
            'Cntry': 'MV'}

CITY_RE = re.compile(r"^(?:(\d{5})\s*,?\s*)?(Mal[e\u00e9][\u2019']?|Hulhumal[e\u00e9][\u2019']?)\s*,?\s*(\d{5})?\s*,?$")
MV_RE = re.compile(r"^Rep(\.|ublic)? of Maldives\.?,?$", re.I)

def parse_contact_lines(lines):
    """address blocks mix address lines with 'Tel:'/'Fax:'/'Email:' labels (value inline or on
    the next line), a bare e-mail (the mailto hrefs are sometimes malformed - use the text),
    a bare website URL, a 'ZIP, City' line and a 'Republic of Maldives' line."""
    out = {'addr': [], 'City': '', 'Zip': '', 'Phone': '', 'Fax': '', 'Email': '', 'Website': ''}
    i = 0
    while i < len(lines):
        l = lines[i]
        ll = l.lower()
        if ll in ('tel:', 'fax:'):
            key = 'Phone' if ll == 'tel:' else 'Fax'
            if i + 1 < len(lines) and not out[key]:
                out[key] = lines[i + 1]
            i += 2
            continue
        if ll.startswith(('tel:', 'tel :')):
            out['Phone'] = l.split(':', 1)[1].strip()
        elif ll.startswith(('fax:', 'fax :')):
            out['Fax'] = l.split(':', 1)[1].strip()
        elif ll.startswith('email:'):
            out['Email'] = l.split(':', 1)[1].strip()
        elif re.match(r'^\S+@\S+$', l):
            out['Email'] = out['Email'] or l
        elif re.match(r'^(https?://|www\.)\S+$', l):
            out['Website'] = out['Website'] or l
        elif MV_RE.match(l):
            pass
        else:
            m = CITY_RE.match(l)
            if m:
                out['Zip'] = m.group(1) or m.group(3) or out['Zip']
                out['City'] = m.group(2).rstrip(',')
            else:
                out['addr'].append(l.rstrip(','))
        i += 1
    out['Address_1'] = ', '.join(out.pop('addr'))
    return out

mother_iso = {'India': 'IN', 'Pakistan': 'PK', 'Srilanka': 'LK', 'Sri Lanka': 'LK',
              'Mauritius': 'MU', 'Bangladesh': 'BD', 'China': 'CN', 'Hong Kong': 'HK'}
UI_JUNK = re.compile(r'^(View|Less Information|More Information)$', re.I)

In [5]:
#------------------------------------------------ Begin_Main : List 1 - Register of Banks ----------------------------------------
listnr = 1
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['Partial'])
count = 0
for p in soup.select('div.panel'):
    h3 = p.find('h3')
    if not h3:
        continue
    name = re.sub(r'\s+', ' ', h3.get_text(' ', strip=True))
    mother = {}
    ho = p.select_one('div[ng-show*="viewDetails"]')   # head-office block, present for foreign branches
    if ho:
        hlines = clean_lines(ho)
        if hlines and hlines[0].lower().startswith('details of head office'):
            hlines = hlines[1:]
        block, mtel = [], ''
        for l in hlines:
            if l.lower().startswith(('tel', 'fax')):
                if l.lower().startswith('tel') and not mtel and ':' in l:
                    mtel = l.split(':', 1)[1].strip()
                continue
            if UI_JUNK.match(l) or l.lower().startswith('website:'):
                continue
            block.append(l.rstrip(','))
        if block:
            mother['Name - Mother Company'] = block[0]
            rest = block[1:]
            if rest:
                cand = re.sub(r'^(Republic|Rep\.?) of ', '', rest[-1])
                if cand in mother_iso:
                    mother['Cntry - Mother company'] = mother_iso[cand]
                    rest = rest[:-1]
            mother['Address_1 - Mother company'] = ', '.join(rest)
            mother['Phone - Mother company'] = mtel
        ho.extract()
    addrs = p.find_all('address')
    fields = parse_contact_lines(clean_lines(addrs[0])) if addrs else {}
    row = common_fields(listnr)
    row.update(fields)
    row.update(mother)
    row['Name'] = name
    add_row(row)
    count += 1

print(f"[INFO] : List 1 -> {count} entities")

[INFO] : Working _(Register of Banks)_ 


[INFO] : List 1 -> 9 entities


In [6]:
#------------------------------------------------ Begin_Main : List 3 - Register of Insurance Providers ----------------------------------------
listnr = 3
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['Partial'])
count = 0
for p in soup.select('div.insurance-panel'):
    h4 = p.find('h4')
    a = h4.find('a')
    name = re.sub(r'\s+', ' ', h4.get_text(' ', strip=True))
    website = a['href'] if a and a.get('href', '').startswith('http') else ''
    h4.extract()
    fields = parse_contact_lines(clean_lines(p.find('address') or p))
    row = common_fields(listnr)
    row.update(fields)
    row.update({'Name': name, 'Website': website or fields.get('Website', '')})
    add_row(row)
    count += 1

print(f"[INFO] : List 3 -> {count} entities")

[INFO] : Working _(Register of Insurance Providers)_ 


[INFO] : List 3 -> 5 entities


In [7]:
#------------------------------------------------ Begin_Main : List 4 - Register of Insurance Brokers (PDF) ----------------------------------------
listnr = 4
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

# the PDF filename is date-stamped - resolve the current link from the insurance-sector partial
sector = get_soup(regdict[listnr]['Partial'])
pdf_href = None
for a in sector.find_all('a', href=True):
    if re.sub(r'\s+', ' ', a.get_text(' ', strip=True)) == 'Register of Insurance Brokers':
        pdf_href = a['href']
        break
if pdf_href is None:
    raise RuntimeError('brokers PDF link not found on insurance-sector partial')
pdf_url = urljoin(BASE, pdf_href)
local = os.path.join(tempfolder, 'insurance-brokers.pdf')
r = requests.get(pdf_url, headers=HEADERS, timeout=60, verify=False)
r.raise_for_status()
with open(local, 'wb') as f:
    f.write(r.content)
print(f"[INFO] : downloaded {pdf_url}")

with pdfplumber.open(local) as pdf:
    table = pdf.pages[0].extract_tables()[0]

validity = ''
rows_out = []
for r_ in table:
    c0 = (r_[0] or '').strip()
    if c0.lower().startswith('updated on'):   # footer row -> ListValidityDate
        validity = datetime.datetime.strptime(c0[len('Updated on'):].strip(), '%d %B %Y').strftime('%Y-%m-%d')
        continue
    if not r_[1] or r_[1].strip() in ('', 'Name'):   # title / header rows
        continue
    contact = (r_[4] or '').replace('-\n', '').replace('\n', ' ')   # rejoin wrapped e-mails
    tel = re.search(r'Telephone:\s*(.*?)(?=Fax:|Email:|$)', contact)
    fax = re.search(r'Fax:\s*(.*?)(?=Email:|$)', contact)
    em = re.search(r'Email:\s*(.*)$', contact)
    addr = re.sub(r'\s+', ' ', (r_[5] or '').replace('\n', ', '))
    addr = re.sub(r",?\s*Rep\.? of Maldives\.?$", '', addr).strip().rstrip(',')
    mc = re.search(r"Mal[e\u00e9][\u2019']?", addr)
    rows_out.append({'Name': r_[1].replace('\n', ' ').strip(),
                     'Phone': tel.group(1).strip() if tel else '',
                     'Fax': fax.group(1).strip() if fax else '',
                     'Email': em.group(1).strip() if em else '',
                     'Address_1': addr, 'City': mc.group(0) if mc else ''})
for data in rows_out:
    row = common_fields(listnr)
    row.update(data)
    row['ListValidityDate'] = validity
    add_row(row)

print(f"[INFO] : List 4 -> {len(rows_out)} entities (register updated {validity})")

[INFO] : Working _(Register of Insurance Brokers)_ 


[INFO] : downloaded https://www.mma.gov.mv/files/financialstability/insurance-brokers-03052026.pdf
[INFO] : List 4 -> 12 entities (register updated 2026-04-21)


In [8]:
#------------------------------------------------ Begin_Main : List 5 - Register of Licensed Payment Service Providers ----------------------------------------
listnr = 5
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['Partial'])
count = 0
for p in soup.select('div.payments-panel'):
    h4 = p.find('h4')
    name = re.sub(r'\s+', ' ', h4.get_text(' ', strip=True))
    lic = '; '.join(li.get_text(' ', strip=True) for li in p.select('ul li'))
    for ul in p.select('ul'):
        ul.extract()
    for b in p.select('b'):
        b.extract()   # the 'Licensed Payment Service:' label
    h4.extract()
    fields = parse_contact_lines(clean_lines(p.find('address') or p))
    row = common_fields(listnr)
    row.update(fields)
    row.update({'Name': name, 'License_Type': lic})
    add_row(row)
    count += 1

print(f"[INFO] : List 5 -> {count} entities")

[INFO] : Working _(Register of Licensed Payment Service Providers)_ 


[INFO] : List 5 -> 8 entities


In [9]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 34 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/MV MMA/MV MMA SQL Ready 2026-07-13 10.50.46.xlsx
